# Fewer Toffolis, Same Qubits | 1ba197b0 invsqrt + Cutoff-18 DQI + 15,000-Iteration Refit of the 4033-Loader Chemistry

**Where the public frontier stands (2026-11-15, early morning).** The three files that matter:
1. **invsqrt `1ba197b0` (78,673.57).** This is James Barlow's (manwithacat) version of the public 46-qubit line with Asterios Panoras's dead-write fix, one Toffoli lighter than `e46d90c3` (1,478.75 against 1,479.75 Toffolis). On the board one Toffoli of the invsqrt is worth about +0.0053. Asterios Panoras ships it in *QCC 27.3122* (27.3122), and manwithacat in *The Real $150K Was the Friends We Made* (27.31216).
2. **The DQI.** Every top notebook still ships our `821bf8d1` (Syed Asad Ali's v12 decoder at extractor cutoff 18 with our 241 retained products), which is rebuilt here from source.
3. **The chemistry.** These are maoxin123's 4033-Toffoli loaders (`T = 14,728`), and the open question is the weights. Asterios Panoras refits Atakan Aldemir's weights to `lambda_eff` 46.905642 (294,982,831.17). Our version 13 ran prvsiyan's refit for 12,000 iterations from Atakan Aldemir's `e52b7cdd` and reached **46.905456** (294,981,656.94, `764dfcbd`).

**This version** pairs `1ba197b0` with our version-13 weights and continues the refit for 3,000 more iterations, for 15,000 in total. With the version-13 weights alone the portfolio scores 27.31223; section 8 prints the official number after the continuation.

| Task | File | Qubits | Cost (official verifier) |
|---|---|---:|---:|
| invsqrt | `1ba197b0` (manwithacat's line with Asterios Panoras's dead-write fix) | 46 | 78,673.57 |
| DQI | ours, `821bf8d1` | 889 | 360,209,905,486 at zero failed shots |
| **chemistry** | **maoxin123's 4033-Toffoli loaders; weights refit here from Atakan Aldemir's `e52b7cdd` (12,000 iterations in version 13, 3,000 more here)** | 427 | **section 7 table** |

The geometric mean is computed from the official verifiers in section 8. The chemistry verifier is deterministic, and the DQI file is byte-identical to the one that has drawn zero failed shots on the public split in every notebook that ships it, so the printed number is the public score.

**What you can take from here**
1. Section 7: prvsiyan's split-L1 reduced-gradient refit run long on a fresh table. Atakan Aldemir's weights were not a local optimum: 12,000 iterations took `lambda_eff` from 46.905894 to 46.905456, and it was still falling.
2. The DQI everyone ships, rebuilt from Syed Asad Ali's source with the cutoff ladder, product retention with its exhaustive proof harness, the paired replay and the audits (sections 2 to 6).

The circuits are Syed Asad Ali's, Ali Haydar Özdağ's, Atakan Aldemir's, James Barlow's, Asterios Panoras's and maoxin123's. The start weights are Atakan Aldemir's, refit here with prvsiyan's code. The cutoff idea is Ertuğrul Özer's, and the retained products are ours. Full credits are at the end.


In [1]:
import sys, os, pathlib, zipfile, json, time, subprocess, re, shutil, hashlib, math, collections, random, itertools, io
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
COMP = pathlib.Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
KM = COMP / "kickmix"; VER = COMP / "verifiers"
sys.path.insert(0, str(KM)); sys.path.insert(0, str(VER))
import kickmix as km, pandas as pd
inp = pathlib.Path("/kaggle/input")
SRC = next(p.parent for p in inp.rglob("SHA256SUMS") if (p.parent / "dqi_source_zip").exists())
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
ENV = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
def verify(args, timeout=5 * 3600):
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=ENV, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stdout": r.stdout[-1500:], "stderr": r.stderr[-1500:]}
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
R0, R1, R2 = 1.1912604648115832, 1.1967314991687559, 1.2013881
DQI_V = VER / "dqi/verify_submission_dqi.py"
KEYS = ("status", "score", "toffoli_count", "max_toffoli_count", "num_qubits", "value_errors", "phase_errors", "repetition_factor")

# The author's published files, checked against his SHA256SUMS; the invsqrt verifier (the slow one) starts now.
SUMS = {}
for line in (SRC / "SHA256SUMS").read_text().splitlines():
    toks = line.replace("*", " ").split(); h = next((t for t in toks if re.fullmatch(r"[0-9a-f]{64}", t)), None)
    if h: SUMS[" ".join(t for t in toks if t != h)] = h
assert all(sha(SRC / n) == h for n, h in SUMS.items()); log(f"all {len(SUMS)} files in SHA256SUMS match ({SRC})")
SUB = pathlib.Path("/tmp/sub"); shutil.rmtree(SUB, ignore_errors=True); (SUB / "dfthcsos").mkdir(parents=True)
INV_PIN = "1ba197b0"   # manwithacat's (James Barlow) 46-qubit line with Asterios Panoras's dead-write fix, as shipped in Asterios Panoras's QCC 27.3122
INV_SRC = next((p for p in sorted(inp.rglob("*invsqrt.kmb")) if sha(p).startswith(INV_PIN)), None)
if INV_SRC is not None: shutil.copy(INV_SRC, SUB / "invsqrt.kmb")
for zp in ([] if INV_SRC else sorted(inp.rglob("submission.zip"))):
    with zipfile.ZipFile(zp) as z:
        if "invsqrt.kmb" in z.namelist() and hashlib.sha256(z.read("invsqrt.kmb")).hexdigest().startswith(INV_PIN):
            (SUB / "invsqrt.kmb").write_bytes(z.read("invsqrt.kmb")); INV_SRC = zp; break
# 1ba197b0 is not shipped as a file anywhere public: Asterios Panoras builds it in his notebook by applying his dead-write
# fix to manwithacat's invsqrt. We do the same: every attached invsqrt goes through the rewrite (a CCX whose target is next
# touched only by an X-basis measurement HMR becomes "HMR first, then CZ on the controls conditioned on the result"), and
# the file whose rewrite hashes to INV_PIN is used.
import bisect, collections
META_OPS = ("APPEND_TO_REGISTER", "REGISTER", "INFO_TAG", "DEBUG_PRINT")
def dead_writes(lines):
    toks = [l.split() for l in lines]; depth = []; d = 0
    for t in toks:
        if t and t[0] == "PUSH_CONDITION": d += 1
        depth.append(d)
        if t and t[0] == "POP_CONDITION": d -= 1
    refs = collections.defaultdict(list)
    for i, t in enumerate(toks):
        if not t or t[0] in META_OPS: continue
        for x in t[1:]:
            if x.startswith("q"): refs[x].append(i)
    hits = []
    for i, t in enumerate(toks):
        if not t or t[0] != "CCX" or "if" in t or depth[i] > 0: continue
        r = refs[t[3]]; k = bisect.bisect_right(r, i)
        if k >= len(r): continue
        j = r[k]; u = toks[j]
        if u[0] == "HMR" and u[1] == t[3] and "if" not in u and depth[j] == 0 and not any(u[2] in toks[x] for x in range(i + 1, j)):
            hits.append((i, j))
    return hits
def dead_write_fix(data):
    src = pathlib.Path("/tmp/dw_in.kmb"); src.write_bytes(data)
    km.Circuit.from_file(str(src), format="kmb").to_file("/tmp/dw_in.kmx", format="kmx")
    lines = pathlib.Path("/tmp/dw_in.kmx").read_text().splitlines(); n = 0
    while True:
        hits = dead_writes(lines)
        if not hits: break
        drop = {j for i, j in hits}; rep = {}
        for i, j in hits:
            _, a, b, tq = lines[i].split()[:4]
            rep[i] = [lines[j].strip(), f"CZ {a} {b} if {lines[j].split()[2]}"]
        lines = [x for idx, l in enumerate(lines) if idx not in drop for x in rep.get(idx, [l])]; n += len(hits)
    pathlib.Path("/tmp/dw_out.kmx").write_text("\n".join(lines) + "\n")
    km.Circuit.from_file("/tmp/dw_out.kmx", format="kmx").to_file("/tmp/dw_out.kmb", format="kmb")
    return pathlib.Path("/tmp/dw_out.kmb").read_bytes(), n, sum(1 for l in lines if l.startswith("CCX"))
if INV_SRC is None:
    CANDS = [(p, p.read_bytes()) for p in sorted(inp.rglob("*invsqrt*.kmb"))]
    for zp in sorted(inp.rglob("*.zip")):
        try:
            with zipfile.ZipFile(zp) as z:
                CANDS += [(zp, z.read(n)) for n in z.namelist() if n.endswith("invsqrt.kmb")]
        except zipfile.BadZipFile: pass
    seen, FIXED = set(), []
    for src, data in CANDS:
        h0 = hashlib.sha256(data).hexdigest()
        if h0 in seen: continue
        seen.add(h0)
        new, n, ccx = dead_write_fix(data); h1 = hashlib.sha256(new).hexdigest()
        log(f"dead-write candidate {h0[:12]} from {src.parent.name}/{src.name}: {n} rewrites, {ccx} CCX left -> {h1[:12]}")
        FIXED.append((h1.startswith(INV_PIN), n, -ccx, src, new, h0, h1))
    match = [f for f in FIXED if f[0]]
    pick = match[0] if match else max((f for f in FIXED if f[1] > 0), key=lambda f: (f[1] > 0, f[2]), default=None)
    if pick is not None:
        (SUB / "invsqrt.kmb").write_bytes(pick[4]); INV_SRC = pick[3]
        log(f"invsqrt rebuilt with the dead-write fix from {pick[5][:12]}: {pick[6][:12]}", "(matches 1ba197b0)" if pick[0] else "(NO pin match: the official verifier below decides)")
assert INV_SRC is not None, "attach manwithacat's 'VQCC Hedge Wizardry Notebook' and 'QCC: Terminal Readout and Phase Folds' (their outputs carry the invsqrt that Asterios Panoras's fix turns into 1ba197b0)"
log(f"invsqrt {INV_PIN} from {INV_SRC.parent.name}")
# maoxin123's 4033-Toffoli loaders (CC0) with our version-13 weights 764dfcbd (12,000 refit iterations from Atakan Aldemir's
# e52b7cdd), taken from this notebook's own version-13 output; the refit of section 7 continues them.
# Fallbacks: Atakan Aldemir's e52b7cdd, then maoxin123's own 3f864a7a, for the same table.
CHEM_SRC = None
for CHEM_PIN in ("764dfcbd0353", "e52b7cddcecc", "3f864a7aeaa4"):
    for zp in sorted(inp.rglob("submission.zip")):
        with zipfile.ZipFile(zp) as z:
            if "dfthcsos/submission.npz" in z.namelist() and hashlib.sha256(z.read("dfthcsos/submission.npz")).hexdigest().startswith(CHEM_PIN):
                for n in z.namelist():
                    if n.startswith("dfthcsos/") and not n.endswith("/"): (SUB / "dfthcsos" / n.split("/")[-1]).write_bytes(z.read(n))
                CHEM_SRC = zp; break
    if CHEM_SRC is not None: break
assert CHEM_SRC is not None, "attach Atakan Aldemir's notebook (output carries e52b7cdd) or maoxin123's 4033 notebook"
LOADERS = {p.name: sha(p)[:12] for p in sorted((SUB / "dfthcsos").glob("angle_k_*.kmb"))}
assert len(LOADERS) == 15
log(f"chemistry start {CHEM_PIN} from {CHEM_SRC.parent.name}; {len(LOADERS)} loaders", LOADERS)
inv_proc = subprocess.Popen([sys.executable, str(VER / "invsqrt/verify_submission_invsqrt.py"), str(SUB / "invsqrt.kmb"), "--max-workers", "2"],
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=ENV, cwd=str(VER))

[  0.1 min] all 23 files in SHA256SUMS match (/kaggle/input/datasets/syed3000/vqcc-own-circuit-portfolio)
[  0.4 min] invsqrt 1ba197b0 from vqcc-hedge-wizardry-notebook
[  0.4 min] chemistry start 764dfcbd0353 from qai-fewer-toffolis-same-qubits-lb-27-30687; 15 loaders {'angle_k_0.kmb': '71735b23037e', 'angle_k_1.kmb': 'e63f9de0a60f', 'angle_k_10.kmb': 'c2e7dca4c51c', 'angle_k_11.kmb': '18fa8cd8125a', 'angle_k_12.kmb': '62d6cd9855bc', 'angle_k_13.kmb': '09a2faee4582', 'angle_k_14.kmb': 'a582eb78249c', 'angle_k_2.kmb': 'e32afaf5d4b0', 'angle_k_3.kmb': 'a932bc254679', 'angle_k_4.kmb': 'dc70bda726f2', 'angle_k_5.kmb': 'dbef059b4cd3', 'angle_k_6.kmb': '11cb19cb7e9d', 'angle_k_7.kmb': 'e3a11ed9c462', 'angle_k_8.kmb': '0fdcfe190658', 'angle_k_9.kmb': 'e0e95ba89402'}


## 1. The metric, as levers

The score is the geometric mean of `reference cost / your cost` over the three tasks. At 27.30, **a 1% cost cut on any one task is worth about +0.09**; a cut of 1e-6 is worth about +0.000009.

| Task | Cost | What 1% is |
|---|---|---|
| invsqrt | `Q x (Toffoli + T/4)` | one qubit is 2.2% (46 qubits) |
| DQI | `R(eps+) x Q^2 x T` | about 3,830 Toffolis per shot; one qubit is 0.225%, about 860 Toffolis |
| chemistry | `lambda_eff x T x Q` | about 148 Toffolis |

The cutoff and the retention together remove about 556 Toffolis per shot from Syed Asad Ali's decoder, 0.15% of the DQI cost; against version 7 the new step is about 105 Toffolis per shot. One failed DQI shot moves the board by about −0.041 (section 9), so the draw matters more than any local rewrite. That is why the paired test in section 5 is there.

## 2. Value fingerprints, idle windows, and the proof harness

- **Value fingerprints.** One pass over the circuit gives every qubit a 64-bit fingerprint of the Boolean function of the inputs it holds. `CX` XORs fingerprints. `CCX` XORs in a fixed random fingerprint per unordered pair of operand fingerprints, after pulling out complements. Anything written under a classical condition or inside a `PUSH_CONDITION` block gets a fresh random fingerprint. Equal fingerprints mean equal functions, up to about `2^-64` per comparison. The rules can miss an equality but never invent one.
- **Idle windows.** The same pass records, for every scratch qubit, the stretches where it is provably `|0>` (fingerprint 0) and untouched.
- **Retention.** If a product is computed twice and some scratch qubit is idle over the whole stretch in between, the first Toffoli also writes the product into that qubit (`CCX a b s ; CX s t`). The second Toffoli becomes `CX s t'`. The host is released by measurement (`HMR s m ; CZ a b if m`), the standard measurement-based uncomputation, which costs no Toffoli. Intervals are scheduled greedily by end time.

The harness below is the proof obligation. It builds thousands of small random circuits rich in the patterns that matter (repeated products, measurements, conditional gates, `PUSH_CONDITION` blocks, swaps, controlled swaps), rewrites them with the planner (with and without extra hosts and moves), and compares the original and the rewrite with a reference simulator of the kickmix semantics, **on every input and every combination of measurement outcomes**. The joint distribution of final qubit values, classical record and phase must match exactly.

In [2]:
%%writefile /tmp/retain.py
import random, collections, bisect

def read_registers(path):
    regq = set()
    with open(path) as f:
        for line in f:
            if line[0] == 'A':
                p = line.split()
                if p[1][0] == 'q': regq.add(p[1])
    return regq

def scan(path, regq, seed=20261112):
    """Forward pass with value fingerprints. Returns
    occ : product fingerprint -> [(line, a, b, t)] for unconditional top-level CCX gates
    idle: qubit -> [(z, L)] maximal windows in which the qubit is provably |0> and untouched (after line z, before line L)
    nlines"""
    rnd = random.Random(seed).getrandbits
    ONE = 1 << 64; LOW = ONE - 1
    h = collections.defaultdict(int)
    for q in regq: h[q] = rnd(64)
    memo = {}; occ = collections.defaultdict(list); idle = collections.defaultdict(list)
    zs = {}                      # qubit -> line after which it has been zero and untouched (top level only)
    seen = set()
    depth = 0; written = []; block_start = [0]
    def AND(x, y):
        xn, x0, yn, y0 = x >> 64, x & LOW, y >> 64, y & LOW
        if x0 == 0 or y0 == 0 or x0 == y0:
            core = x0 if (x0 == y0) else 0; free = True
        else:
            k = (x0, y0) if x0 < y0 else (y0, x0)
            core = memo.get(k)
            if core is None: core = memo[k] = rnd(64)
            free = False
        return core ^ (y0 if xn else 0) ^ (x0 if yn else 0) ^ (ONE if (xn and yn) else 0), free
    def touch(q, L):
        z = zs.pop(q, None)
        if z is not None:
            ins = (L - 1) if depth == 0 else (block_start[0] - 1)      # last top-level point before the touch
            if ins >= z: idle[q].append((z, L, ins))
    def settle(q, L):            # after an unconditional top-level op on q
        if depth == 0 and h[q] == 0 and q not in regq: zs[q] = L
        if written: written[-1].add(q)
    lineno = -1
    with open(path) as f:
        for lineno, line in enumerate(f):
            c0 = line[0]
            if c0 == 'C' and line[1] == 'X':
                p = line.split()
                if len(p) == 3:
                    a = p[1]; b = p[2]
                    if a in zs: touch(a, lineno); zs[a] = lineno
                    if b in zs: touch(b, lineno)
                    v = h[b] ^ h[a]; h[b] = v
                    if depth == 0:
                        if not v and b not in regq: zs[b] = lineno
                    else: written[-1].add(b)
                    continue
            p = line.split()
            if not p: continue
            op = p[0]
            if op in ('INFO_TAG', 'APPEND_TO_REGISTER', 'REGISTER', 'DEBUG_PRINT', 'BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1', 'NEG'): continue
            if op == 'PUSH_CONDITION':
                if depth == 0: block_start[0] = lineno
                depth += 1; written.append(set()); continue
            if op == 'POP_CONDITION':
                if depth:
                    depth -= 1; w = written.pop()
                    for q in w: h[q] = rnd(64); zs.pop(q, None)
                    if written: written[-1] |= w
                continue
            cond = None
            if len(p) >= 3 and p[-2] == 'if': cond = p[-1]; p = p[:-2]
            qs = [t for t in p[1:] if t[0] == 'q']
            zero_before = {q: (q in zs) for q in qs}
            for q in qs: touch(q, lineno)
            if op in ('Z', 'CZ', 'CCZ', 'Z_POW'):
                if cond is None and depth == 0:
                    for q in qs:
                        if zero_before[q]: zs[q] = lineno          # phase gates leave |0> untouched in value; start a new window
                continue
            if cond is not None:
                for q in (qs[-1:] if op in ('X', 'CX', 'CCX', 'HMR', 'R') else qs if op in ('SWAP',) else qs[1:]):
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
                for q in qs:
                    if zero_before[q] and h[q] == 0 and depth == 0: zs[q] = lineno
                continue
            if op == 'X': t = qs[0]; h[t] ^= ONE; settle(t, lineno)
            elif op == 'CX':
                h[qs[1]] ^= h[qs[0]]; settle(qs[1], lineno); settle(qs[0], lineno)
            elif op == 'SWAP':
                a, b = qs; h[a], h[b] = h[b], h[a]; settle(a, lineno); settle(b, lineno)
            elif op in ('R', 'HMR'):
                t = qs[0]; h[t] = 0; settle(t, lineno)
            elif op == 'CCX':
                a, b, t = qs
                P, free = AND(h[a], h[b])
                if not free and P and depth == 0: occ[P].append((lineno, a, b, t))
                h[t] ^= P; settle(t, lineno); settle(a, lineno); settle(b, lineno)
            elif op == 'CSWAP':
                c, a, b = qs
                P, free = AND(h[c], h[a] ^ h[b]); h[a] ^= P; h[b] ^= P
                for q in qs: settle(q, lineno)
            else:
                for q in qs:
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
    n = lineno + 1
    for q, z in zs.items():
        if n - 1 >= z: idle[q].append((z, n, n - 1))
    return occ, idle, n

def plan(occ, idle, first_new_bit, n_lines=0, extra=(), max_span=None, max_hops=0):
    """Hold each repeated product in qubits that are provably |0> and untouched between two of its occurrences.
    A host can serve many products one after another; with max_hops > 0 a product may also move from one host to
    another (CX h g; CX g h, with g = 0) when its current host is needed again. Returns (repl, post, stats)."""
    wins = {q: sorted(ws) for q, ws in idle.items()}           # (z, L, ins): zero after line z, untouched before L, last insertion point ins
    for q in extra: wins[q] = [(-1, n_lines + 1, n_lines)]
    xs = set(extra)
    starts = {q: [w[0] for w in ws] for q, ws in wins.items()}
    busy = {q: [] for q in wins}
    def win_at(q, x):                                           # window of q with z < x... qubit zero just after line x-? : need z <= x < L
        i = bisect.bisect_right(starts[q], x) - 1
        if i < 0: return None
        w = wins[q][i]
        return w if x < w[1] else None
    def next_busy(q, x):
        b = busy[q]; i = bisect.bisect_right(b, (x, 1 << 62))
        if i > 0 and b[i - 1][1] >= x: return x - 1            # occupied at x
        return b[i][0] if i < len(b) else 1 << 62
    def reach(q, x):
        """from insertion point x (after line x), how long q can hold: (last line it may still hold through, last hop point)"""
        w = win_at(q, x)
        if w is None: return None
        nb = next_busy(q, x + 1)
        if nb <= x: return None
        hold_end = min(w[1] - 1, nb - 1)                       # holds the value through line hold_end
        hop_pt = min(w[2], nb - 1, w[1] - 1)                   # last insertion point (after that line) on q
        return hold_end, hop_pt
    pairs = []
    for P, lst in occ.items():
        for k in range(len(lst) - 1):
            st, en = lst[k][0], lst[k + 1][0]
            if max_span is None or en - st <= max_span: pairs.append((en, st, P, k))
    pairs.sort()
    host_path = {}; on_extra = 0; post = collections.defaultdict(list); nhops = 0
    for en, st, P, k in pairs:
        # first host: written at line st (the CCX itself), so it must be zero at st: window with z < st, i.e. win_at(q, st-1)
        best = None
        for q in wins:
            r = reach(q, st - 1)
            if r is None: continue
            done = r[0] >= en
            if not done and max_hops == 0: continue
            key = (done, -(q in xs), -(r[0]) if done else r[0])
            if best is None or key > best[0]: best = (key, q, r)
        if best is None: continue
        path = [(best[1], st)]; segs = []; cur, (hold_end, hop_pt) = best[1], best[2]; seg_start = st; ok = hold_end >= en
        while not ok and len(segs) < max_hops:
            if hop_pt < seg_start: break
            cand = None
            for g in wins:
                if g == cur: continue
                r = reach(g, hop_pt)
                if r is None: continue
                key = (r[0] >= en, -(g in xs), -(r[0]) if r[0] >= en else r[0])
                if r[0] <= hold_end and r[0] < en: continue
                if cand is None or key > cand[0]: cand = (key, g, r)
            if cand is None: break
            segs.append((cur, seg_start, hop_pt)); seg_start = hop_pt; cur = cand[1]; hold_end, hop_pt = cand[2]; path.append((cur, seg_start)); ok = hold_end >= en
        if not ok: continue
        segs.append((cur, seg_start, en))
        for q, a_, b_ in segs: bisect.insort(busy[q], (a_, b_))
        for (q1, _, h), (q2, _) in zip(segs[:-1], path[1:]):
            post[h] += [f"CX {q1} {q2}\n", f"CX {q2} {q1}\n"]; nhops += 1
        host_path[(P, k)] = (path[0][0], cur); on_extra += any(q in xs for q, _, _ in segs)
    repl = {}; nb = first_new_bit; saved = 0
    for P, lst in occ.items():
        for k, (L, a, b, t) in enumerate(lst):
            s_in = host_path.get((P, k - 1), (None, None))[1]; s_out = host_path.get((P, k), (None, None))[0]
            if s_in is None and s_out is None: continue
            out = []
            if s_in is not None: out.append(f"CX {s_in} {t}\n"); saved += 1
            else: out += [f"CCX {a} {b} {s_out}\n", f"CX {s_out} {t}\n"]
            if s_in is not None and s_out is not None: out.append(f"CX {s_in} {s_out}\n")
            if s_in is not None:
                out += [f"HMR {s_in} b{nb}\n", f"CZ {a} {b} if b{nb}\n"]; nb += 1
            repl[L] = out
    stats = dict(duplicate_pairs=sum(len(l) - 1 for l in occ.values() if len(l) > 1), pairs_considered=len(pairs), hosted=len(host_path), hops=nhops,
                 on_extra=on_extra, extra_qubits=len(xs), toffolis_saved=saved, new_bits=nb - first_new_bit)
    return repl, post, stats

def rewrite(src, dst, repl, post):
    with open(src) as f, open(dst, 'w') as g:
        for i, line in enumerate(f):
            if i in repl: g.writelines(repl[i])
            else: g.write(line)
            if i in post: g.writelines(post[i])

Writing /tmp/retain.py


In [3]:
%%writefile /tmp/peep.py
import random, collections

def read_registers(path):
    """Qubits that belong to a register (inputs / outputs): their value is unknown at the start."""
    regq = set()
    with open(path) as f:
        for line in f:
            if line[0] == 'A':
                p = line.split()
                if p[1][0] == 'q': regq.add(p[1])
    return regq

def scan(path, regq, first_new_bit, seed=20261112):
    """One forward pass. Returns (repl, pre, post, stats, examples).
    repl[lineno] = list of lines replacing that line; pre/post[lineno] = lines inserted before/after."""
    rnd = random.Random(seed).getrandbits
    ONE = 1 << 64; LOW = ONE - 1
    h = collections.defaultdict(int); ver = collections.defaultdict(int)
    for q in regq: h[q] = rnd(64)
    pend = {}; csw = {}; await_bits = {}
    memo = {}; home = {}
    repl = {}; pre = collections.defaultdict(list); post = collections.defaultdict(list)
    stats = collections.Counter(); examples = []
    frames = [0]; nframe = 0; written = []      # written[i] = set of qubits written in open block i
    nb = [first_new_bit]; tag = ['']

    def note(kind, lineno, text):
        stats[kind] += 1
        if len(examples) < 400: examples.append((kind, lineno, tag[0], text))

    def finalize(t, m, hline):
        """An X-basis measurement of t (bit m) at line hline: every still-valid pending Toffoli on t is dropped."""
        cands = pend.pop(t, None)
        if not cands: return 0
        fr = frames[-1]; n = 0
        for (L, a, b, va, vb, cfr) in cands:
            if cfr == fr and ver[a] == va and ver[b] == vb and L not in repl:
                repl[L] = []; post[hline].append(f"CZ {a} {b} if {m}\n"); n += 1
                note('R4_ccx_then_measure', L, f"CCX {a} {b} {t} -> CZ {a} {b} if {m} @line {hline}")
        return n

    def zero_event(t, lineno):
        """t is provably |0> here and has pending Toffolis: a free HMR lets them be dropped."""
        fr = frames[-1]
        if not any(c[5] == fr and ver[c[1]] == c[3] and ver[c[2]] == c[4] and c[0] not in repl for c in pend.get(t, ())):
            pend.pop(t, None); return
        m = f"b{nb[0]}"; nb[0] += 1
        post[lineno].append(f"HMR {t} {m}\n")
        finalize(t, m, lineno)

    def AND(x, y):
        xn, x0, yn, y0 = x >> 64, x & LOW, y >> 64, y & LOW
        if x0 == 0 or y0 == 0 or x0 == y0:
            core = x0 if (x0 == y0) else 0; free = True
        else:
            k = (x0, y0) if x0 < y0 else (y0, x0)
            core = memo.get(k)
            if core is None: core = memo[k] = rnd(64)
            free = False
        P = core ^ (y0 if xn else 0) ^ (x0 if yn else 0) ^ (ONE if (xn and yn) else 0)
        return P, free

    def kill_rec(r):
        r['dead'] = True
        for o in (r['a'], r['b']):
            if csw.get(o) is r: del csw[o]
        m1 = r.get('m1')
        if m1 is not None and await_bits.get(m1) is r: del await_bits[m1]

    def kill_csw(q):
        r = csw.get(q)
        if r is not None: kill_rec(r)

    def use(q):            # q read as a control or Z-type operand
        if q in pend: del pend[q]
        if q in csw: kill_csw(q)

    def wrote(q):
        ver[q] += 1
        if written: written[-1].add(q)
        if q in csw: kill_csw(q)

    def fresh(q):
        h[q] = rnd(64); wrote(q)

    with open(path) as f:
        for lineno, line in enumerate(f):
            c0 = line[0]
            if c0 == 'C' and line[1] == 'X':
                p = line.split()
                if len(p) == 3:
                    a = p[1]; b = p[2]
                    v = h[b] ^ h[a]; h[b] = v; ver[b] += 1
                    if a in pend: del pend[a]
                    if written: written[-1].add(b)
                    if csw:
                        if a in csw: kill_csw(a)
                        if b in csw: kill_csw(b)
                    if not v and b in pend: zero_event(b, lineno)
                    continue
            p = line.split()
            if not p: continue
            op = p[0]
            if op == 'INFO_TAG': tag[0] = line[9:].strip()[:80]; continue
            if op in ('APPEND_TO_REGISTER', 'REGISTER', 'DEBUG_PRINT'): continue
            cond = None
            if len(p) >= 3 and p[-2] == 'if': cond = p[-1]; p = p[:-2]
            if await_bits:
                for tok in line.split()[1:]:
                    r = await_bits.get(tok)
                    if r is not None: kill_rec(r)
            if op == 'PUSH_CONDITION':
                nframe += 1; frames.append(nframe); written.append(set()); continue
            if op == 'POP_CONDITION':
                if len(frames) > 1:
                    frames.pop(); w = written.pop()
                    for q in w:
                        h[q] = rnd(64); ver[q] += 1
                        if q in csw: kill_csw(q)
                    if written: written[-1] |= w
                continue
            if op in ('BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1', 'NEG'): continue
            if op in ('Z', 'CZ', 'CCZ', 'Z_POW'):
                qs = [t for t in p[1:] if t[0] == 'q']
                if op == 'CCZ' and cond is None:
                    vals = [h[q] for q in qs]
                    if 0 in vals:
                        repl[lineno] = []; note('R5_ccz_zero_operand', lineno, line.strip())
                for q in qs: use(q)
                continue
            if op == 'X':
                t = p[1]
                if cond is None:
                    h[t] ^= ONE; wrote(t)
                else: fresh(t)
                continue
            if op == 'CX':            # conditioned CX
                use(p[1]); fresh(p[2]); continue
            if op == 'SWAP':
                a, b = p[1], p[2]; use(a); use(b)
                if cond is None:
                    h[a], h[b] = h[b], h[a]; wrote(a); wrote(b)
                else: fresh(a); fresh(b)
                continue
            if op in ('R', 'HMR'):
                t = p[1]
                if cond is not None:
                    pend.pop(t, None); fresh(t); continue
                if op == 'R':
                    pend.pop(t, None)
                    if t in csw: kill_csw(t)
                    h[t] = 0; ver[t] += 1
                    if written: written[-1].add(t)
                    continue
                m = p[2]
                r = csw.get(t)
                if r is not None and not r.get('dead'):
                    a, b, c = r['a'], r['b'], r['c']
                    ok = (r['fr'] == frames[-1] and ver[c] == r['vc'])
                    if r['state'] == 0:
                        o = b if t == a else a
                        ok = ok and ver[a] == r['va'] and ver[b] == r['vb'] and a in pend and b in pend
                        if ok:
                            r.update(state=1, first=t, other=o, m1=m, l1=lineno, expect=o, vo=ver[o])
                            await_bits[m] = r
                            del csw[t]; pend.pop(t, None)
                            h[t] = 0; ver[t] += 1
                            if written: written[-1].add(t)
                            continue
                        kill_csw(a); kill_csw(b)
                    else:
                        ok = ok and t == r['other'] and ver[t] == r['vo'] and t in pend and m != r['m1']
                        if ok:
                            f1, m1, l1, L = r['first'], r['m1'], r['l1'], r['line']
                            repl[L] = []
                            pre[l1].append(f"CX {f1} {t}\n"); post[l1].append(f"CZ {c} {t} if {m1}\n")
                            post[lineno].append(f"BIT_INVERT {m1} if {m}\n")
                            note('R3_cswap_window', L, f"CSWAP {c} {a} {b} measured at lines {l1},{lineno}")
                            del csw[t]; await_bits.pop(m1, None); pend.pop(t, None); use(c)
                            h[t] = 0; ver[t] += 1
                            if written: written[-1].add(t)
                            continue
                        kill_csw(t)
                finalize(t, m, lineno)
                h[t] = 0; ver[t] += 1
                if written: written[-1].add(t)
                continue
            if op == 'CCX':
                a, b, t = p[1], p[2], p[3]
                use(a); use(b)
                if cond is not None:
                    fresh(t); continue
                x, y = h[a], h[b]
                P, free = AND(x, y)
                if t in csw: kill_csw(t)
                if free:
                    x0, y0 = x & LOW, y & LOW
                    if P == 0: new = []
                    elif P == x: new = [f"CX {a} {t}\n"]
                    elif P == y: new = [f"CX {b} {t}\n"]
                    else: new = None
                    if new is not None:
                        repl[lineno] = new; note('R5_ccx_trivial', lineno, line.strip() + ' -> ' + (new[0].strip() if new else 'delete'))
                        if new: pend.pop(a if P == x else b, None)
                elif P:
                    q = home.get(P)
                    if q is not None and q != t and h[q] == P:
                        repl[lineno] = [f"CX {q} {t}\n"]; use(q)
                        note('R1_shared_product', lineno, line.strip() + f' -> CX {q} {t}')
                    else:
                        pend.setdefault(t, []).append((lineno, a, b, ver[a], ver[b], frames[-1]))
                        if h[t] == 0: home[P] = t
                v = h[t] ^ P; h[t] = v; ver[t] += 1
                if written: written[-1].add(t)
                if not v and t in pend: zero_event(t, lineno)
                continue
            if op == 'CSWAP':
                c, a, b = p[1], p[2], p[3]
                use(c); use(a); use(b)
                if cond is not None:
                    fresh(a); fresh(b); continue
                d = h[a] ^ h[b]
                P, free = AND(h[c], d)
                if P == 0:
                    repl[lineno] = []; note('R5_cswap_noop', lineno, line.strip())
                    continue
                h[a] ^= P; h[b] ^= P; ver[a] += 1; ver[b] += 1
                if written: written[-1].add(a); written[-1].add(b)
                r = dict(line=lineno, c=c, a=a, b=b, va=ver[a], vb=ver[b], vc=ver[c], fr=frames[-1], state=0)
                csw[a] = r; csw[b] = r; pend[a] = []; pend[b] = []
                continue
            stats['unknown_op_' + op] += 1
            for tkn in p[1:]:
                if tkn[0] == 'q': use(tkn); fresh(tkn)
    stats['memo_products'] = len(memo); stats['new_bits'] = nb[0] - first_new_bit
    return repl, pre, post, stats, examples

def rewrite(src, dst, repl, pre, post):
    with open(src) as f, open(dst, 'w') as g:
        w = g.write
        for i, line in enumerate(f):
            if i in pre: g.writelines(pre[i])
            if i in repl: g.writelines(repl[i])
            else: w(line)
            if i in post: g.writelines(post[i])

Writing /tmp/peep.py


In [4]:
%%writefile /tmp/alloc.py
"""Scratch-qubit reallocation for kickmix circuits.

A scratch qubit is *idle* on a line if it is provably |0> there (value fingerprint 0) and no instruction touches it.
Between idle stretches it is *busy*. Two busy segments that do not overlap in time can live on the same physical wire,
because the wire is |0> at both ends of each segment. Interval colouring gives the fewest wires: the peak number of
simultaneously busy scratch qubits."""
import random, collections, re

def read_registers(path):
    regq = set()
    with open(path) as f:
        for line in f:
            if line[0] == 'A':
                p = line.split()
                if p[1][0] == 'q': regq.add(p[1])
    return regq

def busy_segments(path, regq, seed=20261112):
    """Returns (segments, n_lines, qubits): segments[q] = list of [start, end] line ranges (inclusive) where scratch
    qubit q is not provably idle."""
    rnd = random.Random(seed).getrandbits
    ONE = 1 << 64; LOW = ONE - 1
    h = collections.defaultdict(int)
    for q in regq: h[q] = rnd(64)
    memo = {}
    zero_since = {}            # scratch qubit -> line after which it is |0> and untouched (top level), -1 = from the start
    seg_open = {}              # scratch qubit -> start line of the current busy segment
    segs = collections.defaultdict(list); seen = set()
    depth = 0; written = []
    def AND(x, y):
        xn, x0, yn, y0 = x >> 64, x & LOW, y >> 64, y & LOW
        if x0 == 0 or y0 == 0 or x0 == y0:
            core = x0 if (x0 == y0) else 0
        else:
            k = (x0, y0) if x0 < y0 else (y0, x0)
            core = memo.get(k)
            if core is None: core = memo[k] = rnd(64)
        return core ^ (y0 if xn else 0) ^ (x0 if yn else 0) ^ (ONE if (xn and yn) else 0)
    def touch(q, L):
        if q in regq: return
        if q not in seen: seen.add(q); zero_since[q] = -1
        if q in zero_since:
            del zero_since[q]; seg_open[q] = L
    def settle(q, L):
        if q in regq: return
        if written: written[-1].add(q)
        if depth == 0 and h[q] == 0 and q in seg_open:
            segs[q].append([seg_open.pop(q), L]); zero_since[q] = L
    lineno = -1
    with open(path) as f:
        for lineno, line in enumerate(f):
            p = line.split()
            if not p: continue
            op = p[0]
            if op in ('INFO_TAG', 'APPEND_TO_REGISTER', 'REGISTER', 'DEBUG_PRINT', 'BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1', 'NEG'): continue
            if op == 'PUSH_CONDITION': depth += 1; written.append(set()); continue
            if op == 'POP_CONDITION':
                if depth:
                    depth -= 1; w = written.pop()
                    for q in w: h[q] = rnd(64)
                    if written: written[-1] |= w
                continue
            cond = None
            if len(p) >= 3 and p[-2] == 'if': cond = p[-1]; p = p[:-2]
            qs = [t for t in p[1:] if t[0] == 'q']
            for q in qs: touch(q, lineno)
            if op in ('Z', 'CZ', 'CCZ', 'Z_POW'):
                for q in qs: settle(q, lineno)
                continue
            if cond is not None or depth:
                tgt = qs[-1:] if op in ('X', 'CX', 'CCX', 'HMR', 'R') else qs if op == 'SWAP' else qs[1:]
                if cond is not None:
                    for q in tgt: h[q] = rnd(64)
                else:
                    # inside a block but unconditional: track values; the block's writes are randomised at POP
                    pass
                if cond is not None:
                    for q in qs: settle(q, lineno)
                    continue
            if op == 'X': h[qs[0]] ^= ONE
            elif op == 'CX': h[qs[1]] ^= h[qs[0]]
            elif op == 'SWAP': a, b = qs; h[a], h[b] = h[b], h[a]
            elif op in ('R', 'HMR'): h[qs[0]] = 0
            elif op == 'CCX': h[qs[2]] ^= AND(h[qs[0]], h[qs[1]])
            elif op == 'CSWAP':
                c, a, b = qs; P = AND(h[c], h[a] ^ h[b]); h[a] ^= P; h[b] ^= P
            else:
                for q in qs: h[q] = rnd(64)
            for q in qs: settle(q, lineno)
    n = lineno + 1
    for q, s in seg_open.items(): segs[q].append([s, n])          # never returned to |0>: busy to the end
    qubits = sorted(seen | set(regq), key=lambda t: int(t[1:]))
    return segs, n, qubits

def colour(segs):
    """Greedy interval colouring by start time (optimal for interval graphs). Returns (assign, n_colours, peak)
    where assign[(q, i)] = colour of segment i of qubit q."""
    events = sorted(((s, e, q, i) for q, ss in segs.items() for i, (s, e) in enumerate(ss)))
    import heapq
    free, busy, assign, n_col = [], [], {}, 0
    for s, e, q, i in events:
        while busy and busy[0][0] < s:
            _, c = heapq.heappop(busy); heapq.heappush(free, c)
        if free: c = heapq.heappop(free)
        else: c = n_col; n_col += 1
        assign[(q, i)] = c; heapq.heappush(busy, (e, c))
    return assign, n_col

def rewrite(src, dst, regq, segs, assign, qubits):
    """Relabel every scratch-qubit occurrence by the colour of the busy segment it falls in, then renumber all qubits
    contiguously (register qubits keep their relative order first)."""
    import bisect
    starts = {q: [s for s, _ in ss] for q, ss in segs.items()}
    reg_sorted = sorted(regq, key=lambda t: int(t[1:]))
    n_col = (max(assign.values()) + 1) if assign else 0
    # final names: keep register qubits at their original indices if possible; colours fill the remaining lowest indices
    reg_idx = {int(q[1:]) for q in reg_sorted}
    total = len(reg_sorted) + n_col
    free_idx = [i for i in range(total + len(reg_idx) + 1) if i not in reg_idx]
    # ensure all register indices < total, otherwise renumber registers too
    if reg_idx and max(reg_idx) >= total:
        reg_map = {q: f"q{k}" for k, q in enumerate(reg_sorted)}
        col_name = [f"q{len(reg_sorted) + c}" for c in range(n_col)]
    else:
        reg_map = {q: q for q in reg_sorted}
        col_name = [f"q{i}" for i in free_idx[:n_col]]
    tok = re.compile(r"\bq\d+\b")
    with open(src) as f, open(dst, 'w') as g:
        for L, line in enumerate(f):
            if 'q' not in line: g.write(line); continue
            def sub(m):
                q = m.group(0)
                if q in reg_map: return reg_map[q]
                ss = starts.get(q)
                if not ss: raise ValueError(f"scratch qubit {q} used at line {L} outside any busy segment")
                i = bisect.bisect_right(ss, L) - 1
                if i < 0 or not (segs[q][i][0] <= L <= segs[q][i][1]):
                    raise ValueError(f"{q} used at line {L} outside its busy segments")
                return col_name[assign[(q, i)]]
            g.write(tok.sub(sub, line))
    return total

Writing /tmp/alloc.py


In [5]:
sys.path.insert(0, "/tmp"); import retain, peep, alloc

def run(lines, qin, outcomes):
    """Reference trajectory semantics of kickmix for the instructions used below."""
    q = collections.defaultdict(int, qin); b = collections.defaultdict(int); ph = 0; stack = []; it = iter(outcomes)
    for line in lines:
        p = line.split()
        if not p: continue
        op = p[0]
        if op == "PUSH_CONDITION": stack.append(b[p[1]]); continue
        if op == "POP_CONDITION":
            if stack: stack.pop()
            continue
        if op in ("REGISTER", "APPEND_TO_REGISTER", "INFO_TAG") or not all(stack): continue
        if len(p) >= 3 and p[-2] == "if":
            if not b[p[-1]]: continue
            p = p[:-2]
        a = p[1:]
        if op == "X": q[a[0]] ^= 1
        elif op == "CX": q[a[1]] ^= q[a[0]]
        elif op == "CCX": q[a[2]] ^= q[a[0]] & q[a[1]]
        elif op == "SWAP": q[a[0]], q[a[1]] = q[a[1]], q[a[0]]
        elif op == "CSWAP":
            if q[a[0]]: q[a[1]], q[a[2]] = q[a[2]], q[a[1]]
        elif op == "Z": ph ^= q[a[0]]
        elif op == "CZ": ph ^= q[a[0]] & q[a[1]]
        elif op == "CCZ": ph ^= q[a[0]] & q[a[1]] & q[a[2]]
        elif op == "HMR": m = next(it); ph ^= m & q[a[0]]; q[a[0]] = 0; b[a[1]] = m
        elif op == "BIT_INVERT": b[a[0]] ^= 1
        else: raise ValueError(op)
    return q, b, ph

def dist(lines, qin, nbits, nq):
    nh = sum(l.startswith("HMR") for l in lines); c = collections.Counter()
    for oc in itertools.product((0, 1), repeat=nh):
        q, b, ph = run(lines, qin, oc)
        c[(tuple(q[f"q{i}"] for i in range(nq)), tuple(b[f"b{i}"] for i in range(nbits)), ph)] += 1
    tot = sum(c.values()); return {k: v / tot for k, v in c.items()}

rng = random.Random(5); nq, nreg, nbits = 7, 3, 4; Q = [f"q{i}" for i in range(nq)]; WS = Q[nreg:]
def noise(k):
    L = []
    for _ in range(k):
        qs = rng.sample(Q, 3); r = rng.random(); c = f" if b{rng.randrange(nbits)}" if rng.random() < 0.12 else ""
        if r < 0.30: L.append(f"CX {qs[0]} {qs[1]}{c}\n")
        elif r < 0.40: L.append(f"X {qs[0]}{c}\n")
        elif r < 0.50: L.append(f"CZ {qs[0]} {qs[1]}{c}\n")
        elif r < 0.62: L.append(f"CCX {qs[0]} {qs[1]} {qs[2]}{c}\n")
        elif r < 0.68: L.append(f"BIT_INVERT b{rng.randrange(nbits)}{c}\n")
        elif r < 0.74: L.append(f"SWAP {qs[0]} {qs[1]}{c}\n")
        elif r < 0.80: L.append(f"CSWAP {qs[0]} {qs[1]} {qs[2]}{c}\n")
        elif r < 0.86: L.append(f"PUSH_CONDITION b{rng.randrange(nbits)}\nCX {qs[0]} {qs[1]}\nCCX {qs[2]} {qs[0]} {qs[1]}\nHMR {qs[1]} b{rng.randrange(nbits)}\nPOP_CONDITION\n")
        else: L.append(f"HMR {rng.choice(WS)} b{rng.randrange(nbits)}\n")
    return L
checked = bad = saved = on_extra = moves = 0
for trial in range(14000):
    lines = [f"APPEND_TO_REGISTER q{i} r0\n" for i in range(nreg)] + noise(rng.randrange(0, 3))
    for _ in range(rng.randrange(1, 3)):
        a, b = rng.sample(Q[:nreg + 1], 2)
        for r in range(rng.randrange(2, 4)):
            t = rng.choice([q for q in WS if q not in (a, b)])
            lines.append(f"CCX {a} {b} {t}\n"); lines += noise(rng.randrange(0, 3))
            if rng.random() < 0.7: lines.append(f"HMR {t} b{rng.randrange(nbits)}\n")
            lines += noise(rng.randrange(0, 2))
    lines = [l for l in "".join(lines).splitlines(True) if l.strip()]
    if sum(l.startswith("HMR") for l in lines) > 6: continue
    open("/tmp/t_in.kmx", "w").writelines(lines)
    occ, idle, n = retain.scan("/tmp/t_in.kmx", retain.read_registers("/tmp/t_in.kmx"), seed=trial)
    kx = rng.choice([0, 0, 1, 2]); extra = [f"q{nq + i}" for i in range(kx)]
    repl, post, st = retain.plan(occ, idle, first_new_bit=nbits, n_lines=n, extra=extra, max_hops=rng.choice([0, 3, 8]))
    if not repl: continue
    retain.rewrite("/tmp/t_in.kmx", "/tmp/t_out.kmx", repl, post); new = open("/tmp/t_out.kmx").readlines()
    if sum(l.startswith("HMR") for l in new) > 9: continue
    checked += 1; saved += st["toffolis_saved"]; on_extra += st["on_extra"]; moves += st["hops"]
    for vals in itertools.product((0, 1), repeat=nreg):
        qin = {f"q{i}": v for i, v in enumerate(vals)}
        d0, d1 = dist(lines, qin, nbits, nq + kx), dist(new, qin, nbits, nq + kx)
        if d0.keys() != d1.keys() or any(abs(d0[k] - d1[k]) > 1e-12 for k in d0): bad += 1; break
log(f"self-test: {checked:,} rewritten circuits checked on every input and every measurement outcome; "
    f"{saved:,} Toffolis removed ({on_extra:,} hosted on extra qubits, {moves:,} moves between hosts); mismatches: {bad}")
assert bad == 0

[  4.4 min] self-test: 5,153 rewritten circuits checked on every input and every measurement outcome; 7,462 Toffolis removed (3,560 hosted on extra qubits, 500 moves between hosts); mismatches: 0


## 3. The cutoff ladder, and the cutoff-18 decoder rebuilt from Syed Asad Ali's source

Syed Asad Ali's decoder scans candidate error positions with a prefix emitter whose extractor cutoff is `j_min = 15`. Ertuğrul Özer's change is one runtime wrapper on `P.First3.emit_prefix` that raises it (his notebook ships 17; version 7 of this notebook reproduced his file `cc0d7d25` byte for byte with the script below). Each step up removes controller gates and stops decoding the inputs whose relevant weight falls below the new cutoff.

The table below is the price of each step under the official input law (`optimal_dqi_weights` squared, as the verifier draws them):

| cutoff | excluded probability (vs 15) | expected excluded shots in 102,400 | static Toffoli-like gates |
|---:|---:|---:|---:|
| 15 (Syed Asad Ali) | 0 | 0 | 384,003 |
| 17 (Özer, version 7) | 1.53e-7 | 0.016 | 383,771 |
| **18 (this version)** | **7.66e-7** | **0.078** | **383,655** |
| 19 | 3.46e-6 | 0.354 | not built |

Step 17 → 18 removes 116 more static gates (about +0.003 on the board) for 0.062 more expected excluded shots; step 18 → 19 would cost 0.28 more expected shots for a similar gain, so it is not taken. The cell rebuilds the cutoff-18 decoder and checks it against the file built in our private dry run.

In [6]:
%%writefile /tmp/build_cut.py
# Rebuild Syed Asad Ali's v12 DQI decoder from his published source, with the extractor cutoff moved
# from 15 to CUT (Ertugrul Ozer's runtime wrapper). Usage: build_cut.py SRC KM VER OUT.kmb CUT
import os, sys, pathlib, zipfile, shutil, time
SRC, KM, VER = (pathlib.Path(a) for a in sys.argv[1:4]); OUTP = sys.argv[4]; CUT = int(sys.argv[5])
T0 = time.time()
BUILD = pathlib.Path(f"/tmp/cut_build_{CUT}"); shutil.rmtree(BUILD, ignore_errors=True); BUILD.mkdir(parents=True)
with zipfile.ZipFile(SRC / "dqi_source_zip") as z:
    z.extractall(BUILD)
ROOT = BUILD / "dqi_source" / "src"
OFF = ROOT / "pro1_controller_bundle" / "1_task" / "official"; OFF.mkdir(parents=True, exist_ok=True)
shutil.copy(KM / "kickmix.so", OFF / "kickmix.so")
VDIR = ROOT / "pro1_controller_bundle" / "1_task" / "verifiers" / "dqi"; VDIR.mkdir(parents=True, exist_ok=True)
shutil.copy(VER / "dqi" / "verify_submission_dqi.py", VDIR / "verify_submission_dqi.py")
sys.path.insert(0, str(ROOT / "controller_work" / "after_v3" / "integration_v7"))
os.chdir(ROOT)
import v7_pipeline as P
P.configure()
_orig = P.First3.emit_prefix
def _cut(builder, words, carry, *, j_min=15, **kw):
    return _orig(builder, words, carry, j_min=CUT, **kw)
P.First3.emit_prefix = _cut
import kickmix as km
b = km.CircuitBuilder(); b.set_max_qubits(889)
S = [b.create_quantum_register(12, f"S{i}") for i in range(70)]
E = [b.create_classical_register(12, f"E{i}") for i in range(4095)]
P.emit(b, S, E, j_min=15, scan_positions=4095, stage=lambda m: None)
assert b.num_allocated_qubits == 840, b.num_allocated_qubits
c = b.finish_circuit(); c.to_file(OUTP, format="kmb")
print(f"built cutoff {CUT}: {c.num_qubits} qubits, {c.operation_statistics().num_toffoli_like} static Toffoli-like, {time.time() - T0:.0f}s", flush=True)
shutil.rmtree(BUILD, ignore_errors=True)

Writing /tmp/build_cut.py


In [7]:
r = subprocess.run([sys.executable, "/tmp/build_cut.py", SRC, KM, VER, "/tmp/dqi_cut18.kmb", "18"], capture_output=True, text=True, env=ENV)
print(r.stdout[-600:], r.stderr[-1200:])
DQ17 = pathlib.Path("/tmp/dqi_cut18.kmb"); H17 = sha(DQ17)
log(f"cutoff-18 decoder sha256 {H17[:12]}; byte-identical to the dry-run build 32b0a8b44476: {H17.startswith('32b0a8b44476')}")
assert H17.startswith("32b0a8b44476")
sys.path.insert(0, str(VER / "dqi")); import verify_submission_dqi as vdqi
spec = vdqi.ProblemSpecDQI.benchmark_instance()
prob = vdqi.optimal_dqi_weights(spec.m, spec.max_errors, spec.degree, spec.r) ** 2
cut_df = pd.DataFrame([dict(cutoff=c, excluded_probability_vs_15=float(prob[15:c].sum()),
                            expected_excluded_shots_in_102400=float(prob[15:c].sum()) * spec.num_shots) for c in range(16, 22)])
display(cut_df)

built cutoff 18: 889 qubits, 383655 static Toffoli-like, 321s
 
[  9.7 min] cutoff-18 decoder sha256 32b0a8b44476; byte-identical to the dry-run build 32b0a8b44476: True


,cutoff,excluded_probability_vs_15,expected_excluded_shots_in_102400
0,16,2.444040e-08,0.002503
1,17,1.525103e-07,0.015617
2,18,7.662231e-07,0.078461
3,19,3.461118e-06,0.354418
4,20,1.432082e-05,1.466452
5,21,5.451925e-05,5.582771


## 4. Retention on the cutoff-18 decoder

The planner runs on the cutoff-18 file, with no extra qubit. The table also shows what moves between idle hosts (`max_hops`) and one extra host qubit would add. The relative cost column is `((889 + k) / 889)^2 x (T - saved) / T`.

The rewrite is then applied, and the cell checks that it reproduces, byte for byte, the file built in our private dry run (`821bf8d157c0`).

In [8]:
cd = km.Circuit.from_file(DQ17, format="kmb"); cd.to_file("/tmp/dqi.kmx", format="kmx")
NQ, NB, T_STATIC = cd.num_qubits, cd.num_bits, cd.operation_statistics().num_toffoli_like; del cd
occ, idle, n = retain.scan("/tmp/dqi.kmx", retain.read_registers("/tmp/dqi.kmx"))
spans = sorted(l[k + 1][0] - l[k][0] for l in occ.values() for k in range(len(l) - 1))
log(f"cutoff-18 DQI: {NQ} qubits, {T_STATIC:,} static Toffoli-like gates, {n:,} lines, repeated pairs {len(spans):,}, "
    f"idle windows {sum(len(w) for w in idle.values()):,} on {len(idle)} scratch qubits")
T_MEAN = 382841.36                    # executed Toffolis per shot of the unretained cutoff-18 file (retained mean + the 240.98 per shot the same products save on cutoff 17)
rows, PLANS = [], {}
for k, hops in [(0, 0), (0, 16), (1, 0)]:
    repl, post, st = retain.plan(occ, idle, first_new_bit=NB + 100008, n_lines=n, extra=[f"q{NQ + i}" for i in range(k)], max_hops=hops)
    PLANS[(k, hops)] = (repl, post, st)
    rows.append(dict(extra_qubits=k, max_hops=hops, qubits=NQ + k, toffolis_saved=st["toffolis_saved"], moves_used=st["hops"], hosted_on_extra=st["on_extra"],
                     relative_dqi_cost=round(((NQ + k) / NQ) ** 2 * (T_MEAN - st["toffolis_saved"]) / T_MEAN, 6)))
retention_df = pd.DataFrame(rows); display(retention_df)
del occ, idle
repl, post, st = PLANS[(0, 0)]
retain.rewrite("/tmp/dqi.kmx", "/tmp/dqi_ret.kmx", repl, post)
cr = km.Circuit.from_file("/tmp/dqi_ret.kmx", format="kmx"); cr.to_file("/tmp/dqi_ret.kmb", format="kmb")
T_RET = cr.operation_statistics().num_toffoli_like; HRET = sha("/tmp/dqi_ret.kmb")
log(f"retained DQI: {cr.num_qubits} qubits, {T_RET:,} static Toffoli-like gates ({T_STATIC - T_RET} fewer), sha256 {HRET[:12]}")
assert cr.num_qubits == NQ and T_STATIC - T_RET == st["toffolis_saved"]; del cr
print("byte-identical to the dry-run file:", HRET.startswith("821bf8d157c0"))
want = sorted(set(repl) | set(post))[:6]; orig = {}
with open("/tmp/dqi.kmx") as f:
    for i, line in enumerate(f):
        if i in set(want): orig[i] = line.strip()
        if i > want[-1]: break
print("first rewritten lines:")
for L in want:
    o = orig.get(L, "(end)"); print(f"  line {L:>10,}: {o:<28s} ->", " ; ".join(x.strip() for x in repl.get(L, [o + "\n"]) + post.get(L, [])))
os.remove("/tmp/dqi_ret.kmx"); os.remove("/tmp/dqi.kmx")

[ 11.1 min] cutoff-18 DQI: 889 qubits, 383,655 static Toffoli-like gates, 52,983,602 lines, repeated pairs 5,938, idle windows 168,612 on 49 scratch qubits


,extra_qubits,max_hops,qubits,toffolis_saved,moves_used,hosted_on_extra,relative_dqi_cost
0,0,0,889,241,0,0,0.999370
1,0,16,889,241,0,0,0.999370
2,1,0,890,373,0,132,1.001274


[ 12.1 min] retained DQI: 889 qubits, 383,414 static Toffoli-like gates (241 fewer), sha256 821bf8d157c0
byte-identical to the dry-run file: True
first rewritten lines:
  line 20,920,996: CCX q880 q84 q132            -> CCX q880 q84 q876 ; CX q876 q132
  line 20,920,997: CCX q880 q85 q133            -> CCX q880 q85 q883 ; CX q883 q133
  line 20,920,998: CCX q880 q86 q134            -> CCX q880 q86 q868 ; CX q868 q134
  line 20,920,999: CCX q880 q87 q135            -> CCX q880 q87 q873 ; CX q873 q135
  line 20,921,000: CCX q880 q88 q136            -> CCX q880 q88 q888 ; CX q888 q136
  line 20,921,001: CCX q880 q89 q137            -> CCX q880 q89 q879 ; CX q879 q137


**Reading the table.** The cutoff change only touches the controller prefix, and the retained products sit elsewhere in the decoder, so retention finds the same **241** products on the cutoff-18 file as on cutoff 17 and on the unmodified decoder. Moves between hosts and an extra host qubit add nothing worth a qubit: one qubit costs about 860 Toffolis through `Q^2`. The shipped file uses **no extra qubit**.

## 5. Is the rewritten decoder still right? Official verifier plus a paired replay

The harness in section 2 checks this kind of rewrite exhaustively on small circuits. Here the rewritten 889-qubit decoder runs through the official verifier. Then the **unretained** cutoff-18 file is replayed on exactly the test inputs the rewritten file drew, by passing the rewritten file's own seed to the verifier as `--seed`. If the retained products changed anything on some rare branch, the rewritten file would fail on shots where the unretained one passes.

In [9]:
r_ret = verify([DQI_V, "/tmp/dqi_ret.kmb"])
log("@@VER dqi (cutoff 18 + retention, default salt)", {k: r_ret.get(k) for k in KEYS})
buf = io.BytesIO(); km.Circuit.from_file("/tmp/dqi_ret.kmb", format="kmb").to_file(buf, format="kmb")
SEED = int(hashlib.sha256(buf.getvalue()).hexdigest(), 16)
r_org = verify([DQI_V, DQ17, "--seed", SEED])
log("@@VER dqi (cutoff 18 unretained, on the retained file's inputs)", {k: r_org.get(k) for k in KEYS})
display(pd.DataFrame([dict(decoder="cutoff 18, unretained", **{k: r_org.get(k) for k in KEYS[1:]}),
                      dict(decoder="+ retained products (ours)", **{k: r_ret.get(k) for k in KEYS[1:]})]))
fo = r_org.get("value_errors", 0) + r_org.get("phase_errors", 0); fr = r_ret.get("value_errors", 0) + r_ret.get("phase_errors", 0)
print(f"Same 102,400 inputs: cutoff 18 unretained {fo} failed shots, retained {fr}; Toffolis per shot saved "
      f"{r_org.get('toffoli_count', 0) - r_ret.get('toffoli_count', 0):+.2f}")
assert r_ret.get("status") == "PASS"

[ 15.9 min] @@VER dqi (cutoff 18 + retention, default salt) {'status': 'PASS', 'score': 360209905486.0156, 'toffoli_count': 382600.383203125, 'max_toffoli_count': 383414, 'num_qubits': 889, 'value_errors': 0, 'phase_errors': 0, 'repetition_factor': 1.1912604648115832}
[ 20.0 min] @@VER dqi (cutoff 18 unretained, on the retained file's inputs) {'status': 'PASS', 'score': 360436622906.5201, 'toffoli_count': 382841.1932714844, 'max_toffoli_count': 383655, 'num_qubits': 889, 'value_errors': 0, 'phase_errors': 0, 'repetition_factor': 1.1912604648115832}


,decoder,score,toffoli_count,max_toffoli_count,num_qubits,value_errors,phase_errors,repetition_factor
0,"cutoff 18, unretained",3.604366e+11,382841.193271,383655,889,0,0,1.19126
1,+ retained products (ours),3.602099e+11,382600.383203,383414,889,0,0,1.19126


Same 102,400 inputs: cutoff 18 unretained 0 failed shots, retained 0; Toffolis per shot saved +240.81


## 6. Audit: is anything left in the new decoder?

The same toolkit as in version 6 runs on the rewritten decoder: retention again (with and without moves), five peephole rules, and the interval-colouring bound `min_qubits` (register qubits plus the colouring number of the scratch qubits' busy segments; when it equals `qubits`, no relabelling saves a qubit). It also runs on maoxin123's 15 new chemistry loaders (retention and peephole): a single Toffoli removed there would count twice in the chemistry's `T` and would not touch the DQI's leaderboard draw.

In [10]:
AUD = []
def audit(name, kmx, realloc=True, hops=(0, 16)):
    row = dict(circuit=name)
    try:
        c = km.Circuit.from_file(kmx, format="kmx"); nq, nb = c.num_qubits, c.num_bits
        row.update(qubits=nq, static_toffoli_like=c.operation_statistics().num_toffoli_like); del c
        occ, idle, n = retain.scan(kmx, retain.read_registers(kmx)); row["repeated_pairs"] = sum(max(len(l) - 1, 0) for l in occ.values())
        for h in hops:
            row[f"retained_k0_hops{h}"] = retain.plan(occ, idle, first_new_bit=nb + 100008, n_lines=n, max_hops=h)[2]["toffolis_saved"]
        del occ, idle
        *_, ps, _ = peep.scan(kmx, peep.read_registers(kmx), first_new_bit=nb + 8)
        row["peephole_rewrites"] = sum(v for kk, v in ps.items() if kk[0] == "R")
        if realloc:
            regq = alloc.read_registers(kmx); segs, nl, qs = alloc.busy_segments(kmx, regq); _, ncol = alloc.colour(segs)
            row.update(register_qubits=len(regq), min_qubits=len(regq) + ncol)
    except Exception as e:
        row["skipped"] = repr(e)[:160]
    AUD.append(row); log("@@AUDIT", row)
for p in sorted((SUB / "dfthcsos").glob("angle_k_*.kmb"), key=lambda p: int(p.stem.split("_")[-1])):
    km.Circuit.from_file(p, format="kmb").to_file("/tmp/ld.kmx", format="kmx"); audit(p.stem + " (maoxin123)", "/tmp/ld.kmx", realloc=False, hops=(0,))
os.remove("/tmp/ld.kmx")
km.Circuit.from_file("/tmp/dqi_ret.kmb", format="kmb").to_file("/tmp/dqi_ret.kmx", format="kmx"); audit("dqi (cutoff 18 + retention)", "/tmp/dqi_ret.kmx")
os.remove("/tmp/dqi_ret.kmx")
display(pd.DataFrame(AUD))

[ 20.0 min] @@AUDIT {'circuit': 'angle_k_0 (maoxin123)', 'qubits': 128, 'static_toffoli_like': 291, 'repeated_pairs': 0, 'retained_k0_hops0': 0, 'peephole_rewrites': 0}
[ 20.0 min] @@AUDIT {'circuit': 'angle_k_1 (maoxin123)', 'qubits': 125, 'static_toffoli_like': 286, 'repeated_pairs': 0, 'retained_k0_hops0': 0, 'peephole_rewrites': 0}
[ 20.0 min] @@AUDIT {'circuit': 'angle_k_2 (maoxin123)', 'qubits': 125, 'static_toffoli_like': 286, 'repeated_pairs': 0, 'retained_k0_hops0': 0, 'peephole_rewrites': 0}
[ 20.0 min] @@AUDIT {'circuit': 'angle_k_3 (maoxin123)', 'qubits': 125, 'static_toffoli_like': 286, 'repeated_pairs': 0, 'retained_k0_hops0': 0, 'peephole_rewrites': 0}
[ 20.0 min] @@AUDIT {'circuit': 'angle_k_4 (maoxin123)', 'qubits': 125, 'static_toffoli_like': 263, 'repeated_pairs': 0, 'retained_k0_hops0': 0, 'peephole_rewrites': 0}
[ 20.0 min] @@AUDIT {'circuit': 'angle_k_5 (maoxin123)', 'qubits': 125, 'static_toffoli_like': 263, 'repeated_pairs': 0, 'retained_k0_hops0': 0, 'peephole_

,circuit,qubits,static_toffoli_like,repeated_pairs,retained_k0_hops0,peephole_rewrites,retained_k0_hops16,register_qubits,min_qubits
0,angle_k_0 (maoxin123),128,291,0,0,0,NaN,NaN,NaN
1,angle_k_1 (maoxin123),125,286,0,0,0,NaN,NaN,NaN
2,angle_k_2 (maoxin123),125,286,0,0,0,NaN,NaN,NaN
3,angle_k_3 (maoxin123),125,286,0,0,0,NaN,NaN,NaN
4,angle_k_4 (maoxin123),125,263,0,0,0,NaN,NaN,NaN
5,angle_k_5 (maoxin123),125,263,0,0,0,NaN,NaN,NaN
6,angle_k_6 (maoxin123),125,261,0,0,0,NaN,NaN,NaN
7,angle_k_7 (maoxin123),125,263,0,0,0,NaN,NaN,NaN
8,angle_k_8 (maoxin123),125,261,0,0,0,NaN,NaN,NaN
9,angle_k_9 (maoxin123),125,263,0,0,0,NaN,NaN,NaN


## 7. The chemistry refit on the 4033-Toffoli table

With the loaders fixed, the chemistry cost is `lambda_eff x T x Q` with `T = 14,728` and `Q = 427` fixed, so only `lambda_eff` can move. The verifier decodes the vectors `u` from the fixed angle words and evaluates

- `lambda_eff = sqrt(E_gap (2 Lambda - E_gap))`, with `Lambda = sum_k |eig_k(h1')| + 1/4 sum_(r,c) (sum_b |w_rbc| + |W_rc + sum_b w_rbc|)^2`;
- `h1' = h1 - (eta/2) S + beta_1 I - sum_(r,b) a_rb u_rb u_rb^T`, with `a_rb = sum_c W_rc w_rbc`;
- three gates: `error_1 <= 20 mHa`, `error_2 <= 20 mHa` and `E_gap >= 3.914898 Ha`.

In the file, `W` holds `w` (8 x 143 x 44) and `H2_I` holds `W + sum_b w` (8 x 44). `S` and `beta_1` are not free: the verifier uses the baseline's fixed `H1_I` and `Hsym_lower` and ignores the submitted ones.

**The weights do not transfer between tables.** We compared maoxin123's 4033 file with the refined 4038 files array by array. `H1_I`, `Hsym_lower` and `tree_edges` agree, but the angle words and `U` differ, so the decoded vectors differ. The 4038 table's weights are worth nothing on the new loaders, and the new table needs its own refit.

**The refit.** prvsiyan's `refit_grg.py` (Apache-2.0) splits each weight into non-negative parts, `w = p - n` and `H2_I = h+ - h-`. The absolute values in `Lambda` become linear, and a variable at zero is released whenever the Lagrangian gradient points inward. So the zero set and the signs move with the descent; our version-9 refit, which held them fixed, stalled after -1.7e-8. The direction is an L-BFGS step on the Lagrangian gradient projected onto the tangent space of the active gates. Each trial point is Newton-restored onto the surface, and every saved point is checked with the official `dfthc_math`.

**What we change.** The code runs as published, with three settings changed:
1. **Margins.** `error_2` and `E_gap` sit 3e-10 inside their bounds instead of 3e-9. `error_1` stays 2e-8 inside: its multiplier is about 1e-4 per mHa, so that margin is worth about 1e-9 in `lambda_eff`.
2. **Acceptance.** The script saves a point only if every gate is 1e-9 inside its bound. To match the margins we set this to 1e-10 (the `VM` setting below). A margin of 3e-10 on a 2e-2 gate is 1.5e-8 relative, far above float64 noise.
3. **Length.** Version 13 ran 12,000 iterations from Atakan Aldemir's `e52b7cdd` with smoothing `EPS` 1e-6 and reached `764dfcbd`. This version continues from `764dfcbd` for 3,000 more, with fresh L-BFGS memory. On the 4038 table a second run with a smaller smoothing added about a tenth of what the first did, so all the time goes into one run.

The table below the run gives the official numbers for the start and the result.


In [11]:
# prvsiyan's refit code (Apache-2.0), from his dataset "Circuits That Survived the Salt" (code/refit_grg.py, code/chemlib.py).
PRV = next(p.parent for p in sorted(inp.rglob("refit_grg.py")) if (p.parent / "chemlib.py").exists())
CODE = pathlib.Path("/tmp/prv"); shutil.rmtree(CODE, ignore_errors=True); shutil.copytree(PRV, CODE)
src_grg = (CODE / "refit_grg.py").read_text()
# the one edit: the acceptance threshold of a saved point becomes a setting (VM; the script's own value is 1e-9)
pat = re.compile(r"^(\s*)valid = m\['error_2'\] < 0\.02 - 1e-9 and m\['E_gap'\] > E_GAP_FLOOR \+ 1e-9 and m\['error_1'\] < 0\.02 - 1e-9 and m\['gap_in_domain'\]\s*$", re.M)
assert len(pat.findall(src_grg)) == 1
src_grg = pat.sub(lambda g: g.group(1) + "VM = float(os.environ.get('VM', '1e-9')); valid = m['error_2'] < 0.02 - VM and m['E_gap'] > E_GAP_FLOOR + VM and m['error_1'] < 0.02 - VM and m['gap_in_domain']", src_grg)
(CODE / "refit_grg.py").write_text(src_grg)
log("prvsiyan's refit code from", PRV, "| refit_grg.py", sha(PRV / "refit_grg.py")[:12], "| chemlib.py", sha(PRV / "chemlib.py")[:12])
i0 = src_grg.index('"""'); print(src_grg[i0:src_grg.index('"""', i0 + 3) + 3])


[ 24.7 min] prvsiyan's refit code from /kaggle/input/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a/code | refit_grg.py 58a3e6622213 | chemlib.py fe10126a429d
"""Reduced-gradient refit of the dfthcsos weights (file 'W') and offsets (file 'H2_I') with the tree, angle
words and loaders fixed. Minimises lambda_eff on the surface error_2 = TAU, E_gap = EFL, with error_1 kept
below its bound. Variables are split in non-negative parts (the L1 terms become linear), the search
direction is an L-BFGS direction of the Lagrangian projected on the tangent space of the two active
constraints, and each trial point is put back on the surface by Newton steps along the constraint
gradients. Every reported point is measured with the official metric code (dfthcsos.scoreinfra.dfthc_math).
usage: refit_grg.py IN_NPZ OUT_DIR [iters=2000]   (IN_NPZ needs W, H2_I and the fixed arrays)"""


In [12]:
import numpy as np
KEYS7 = ("error_1", "error_2", "E_gap", "Lambda", "lambda_eff")
chem_start = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
assert chem_start.get("status") == "PASS"
m_start = chem_start["metrics"]
log("start (Atakan Aldemir's e52b7cdd on maoxin123's 4033 loaders):", {k: m_start[k] for k in KEYS7}, "cost", chem_start["score"])
NPZ = dict(np.load(SUB / "dfthcsos/submission.npz"))
START = pathlib.Path("/tmp/chem_start.npz"); shutil.copy(SUB / "dfthcsos/submission.npz", START)

def refit(start_npz, out, iters, **settings):
    """prvsiyan's refit_grg.py as a subprocess; prints one progress line every 250 iterations."""
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    env = dict(os.environ, VQCC_DATA=str(COMP), NT="4", OMP_NUM_THREADS="4", **{k: str(v) for k, v in settings.items()})
    pr = subprocess.Popen([sys.executable, str(CODE / "refit_grg.py"), str(start_npz), str(out), str(iters)], cwd=str(CODE), env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in pr.stdout:
        m = re.match(r"it (\d+) .*official leff ([\d.]+)", line)
        if m and int(m.group(1)) % 250 == 0: log(f"  it {int(m.group(1)):5d}  lambda_eff (smoothed point) {m.group(2)}")
        elif "switched" in line or "failed" in line or line.startswith("done"): log("  " + line.strip()[:200])
    assert pr.wait() == 0
    return pathlib.Path(out) / "best.npz"

def merged(best, path):
    bz = np.load(best); arrs = dict(NPZ)
    arrs["W"] = bz["W"].astype(NPZ["W"].dtype); arrs["H2_I"] = bz["H2_I"].astype(NPZ["H2_I"].dtype)
    np.savez(path, **arrs); return path

STAGES = [dict(stage=f"start: {CHEM_PIN[:8]}", cost=chem_start["score"], **{k: m_start[k] for k in KEYS7})]
best_a = refit(START, "/tmp/runA", 3000, M2="3e-10", ME="3e-10", M1="2e-8", VM="1e-10", EPS="1e-6")
A = merged(best_a, "/tmp/chem_A.npz")
shutil.copy(A, SUB / "dfthcsos/submission.npz")
chem_new = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
assert chem_new.get("status") == "PASS" and chem_new["score"] < chem_start["score"]
m_new = chem_new["metrics"]
STAGES.append(dict(stage="after 3,000 more iterations, margins 3e-10", cost=chem_new["score"], **{k: m_new[k] for k in KEYS7}))
display(pd.DataFrame(STAGES))
REFIT = dict(start_npz=sha(START), new_npz=sha(SUB / "dfthcsos/submission.npz"), code=dict(refit_grg=sha(PRV / "refit_grg.py"), chemlib=sha(PRV / "chemlib.py")),
             stages=STAGES)
log(f"lambda_eff {m_start['lambda_eff']:.10f} -> {m_new['lambda_eff']:.10f} ({m_new['lambda_eff'] - m_start['lambda_eff']:+.3e}); "
    f"chemistry cost {chem_start['score']:,.2f} -> {chem_new['score']:,.2f}; npz {REFIT['new_npz'][:12]}")


[ 25.2 min] start (Atakan Aldemir's e52b7cdd on maoxin123's 4033 loaders): {'error_1': 0.019999979999999123, 'error_2': 0.019999999700000017, 'E_gap': 3.9148980003010365, 'Lambda': 282.95094878218816, 'lambda_eff': 46.90545576783769} cost 294981656.9383006
[ 26.0 min]   it     0  lambda_eff (smoothed point) 46.9054482582
[ 30.6 min]   it   250  lambda_eff (smoothed point) 46.9054468817
[ 35.2 min]   it   500  lambda_eff (smoothed point) 46.9054455223
[ 39.6 min]   it   750  lambda_eff (smoothed point) 46.9054447096
[ 44.1 min]   it  1000  lambda_eff (smoothed point) 46.9054432119
[ 48.7 min]   it  1250  lambda_eff (smoothed point) 46.9054423840
[ 53.2 min]   it  1500  lambda_eff (smoothed point) 46.9054411589
[ 57.7 min]   it  1750  lambda_eff (smoothed point) 46.9054395581
[ 62.2 min]   it  2000  lambda_eff (smoothed point) 46.9054374523
[ 66.8 min]   it  2250  lambda_eff (smoothed point) 46.9054355883
[ 71.4 min]   it  2500  lambda_eff (smoothed point) 46.9054338394
[ 76.0 min]   it 

,stage,cost,error_1,error_2,E_gap,Lambda,lambda_eff
0,start: 764dfcbd,2.949817e+08,0.02,0.02,3.914898,282.950949,46.905456
1,"after 3,000 more iterations, margins 3e-10",2.949815e+08,0.02,0.02,3.914898,282.950719,46.905437


[ 81.2 min] lambda_eff 46.9054557678 -> 46.9054365972 (-1.917e-05); chemistry cost 294,981,656.94 -> 294,981,536.38; npz 4ecd68fc286c


## 8. Official verification of the portfolio and the submission

In [13]:
RES = {"dqi": r_ret}
RES["dfthcsos"] = chem_new
log("@@VER chemistry", {k: RES["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
out, err = inv_proc.communicate()
try: RES["invsqrt"] = json.loads(out[out.index("{"):])
except Exception: RES["invsqrt"] = {"status": "ERROR", "stdout": out[-800:], "stderr": err[-800:]}
log("@@VER invsqrt", {k: RES["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "toffoli_equivalent", "worst_error_over_tolerance")})
assert all(RES[t].get("status") == "PASS" for t in RES)
COST = {"invsqrt": RES["invsqrt"]["score"], "dqi": RES["dqi"]["score"] / RES["dqi"]["repetition_factor"] * R0, "dfthcsos": RES["dfthcsos"]["score"]}
RAT = {t: REF[t] / COST[t] for t in COST}; GM0 = math.prod(RAT.values()) ** (1 / 3)
GM1 = GM0 * (R0 / R1) ** (1 / 3); GM2 = GM0 * (R0 / R2) ** (1 / 3)
display(pd.DataFrame([dict(task=t, qubits=RES[t].get("num_qubits"), cost=COST[t], ratio=RAT[t]) for t in COST]))
log(f"geometric mean at zero failed DQI shots {GM0:.5f}; one failed shot {GM1:.5f}; two {GM2:.5f}")
shutil.copy("/tmp/dqi_ret.kmb", SUB / "dqi.kmb")
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(SUB))
PUB = {str(p.relative_to(SUB)): sha(p) for p in SUB.rglob("*") if p.is_file()}
json.dump({"sha256": PUB, "verify": RES, "paired_cutoff18_unretained": r_org, "retention": retention_df.to_dict("records"), "cutoff_table": cut_df.to_dict("records"),
           "audit": AUD, "chemistry_refit": REFIT, "gm_zero_failures": GM0, "gm_one_failure": GM1, "gm_two_failures": GM2}, open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written:", {k: v[:12] for k, v in PUB.items() if not k.startswith("dfthcsos/")}, "+", len(PUB) - 2, "chemistry files")

[ 81.2 min] @@VER chemistry {'status': 'PASS', 'score': 294981536.37698597, 'toffoli_count': 14728.0, 'num_qubits': 427, 'lambda_eff': 46.90543659721036}
[ 81.2 min] @@VER invsqrt {'status': 'PASS', 'score': 78673.56921101942, 'num_qubits': 46, 'toffoli_count': 1478.750911494735, 'toffoli_equivalent': 1710.2949828482483, 'worst_error_over_tolerance': 0.9986660009092673}


,task,qubits,cost,ratio
0,invsqrt,46,7.867357e+04,26.397356
1,dqi,889,3.602099e+11,67.847839
2,dfthcsos,427,2.949815e+08,11.375626


[ 81.2 min] geometric mean at zero failed DQI shots 27.31224; one failed shot 27.27055; two 27.23527
[ 81.5 min] submission.zip written: {'invsqrt.kmb': '1ba197b02771', 'dqi.kmb': '821bf8d157c0'} + 16 chemistry files


## 9. Reading the badge

- **The draw belongs to the bytes.** The DQI verifier seeds its 102,400 test shots with the file's SHA-256 plus the split's salt. Syed Asad Ali's file draws two failed shots on the public split (27.00209 instead of 27.07839); our version-6, Özer's cutoff-17 and our version-7 files all draw none (27.08407, 27.08334, 27.08903), and so does the version-8 file `821bf8d1` shipped here (27.11042 and 27.29708 here, 27.29709 in Heliosli's notebook, 27.29708 in Atakan Aldemir's and alex chilton's). The chemistry verifier has no random draw, so the public split stays clean for this version.
- **Failures move the score in fixed rungs.** For this portfolio the three rungs (zero, one and two failed shots) are printed in section 8 The two-failure rung uses `R(eps+(2)) / R(eps+(0)) = 1.0085`, pinned by the public badge of Syed Asad Ali's file. On the public split this version scores the zero-failure rung of section 8: the same DQI draw as versions 8 to 10.
- The private split draws again, and no salt can promise it will be clean too.

The method of reading `k` from a badge is Akmal Xodarev's (busyaprime) and James Barlow's (manwithacat); see also prvsiyan's notebook.

## 10. What did not work, and what is left

| Attempt | Result |
|---|---|
| Retention and five peephole rules on maoxin123's 15 coded-word loaders | see the audit table in section 6 (on the v25 loaders: 0 repeated products, 0 rewrites) |
| Moving the symmetry shift `S` (`Hsym_lower`) and `beta_1` (`H1_I`) | the verifier ignores both and uses the baseline's fixed shifts; a dry run that moved `S` looked 1e-4 better on paper and is worth nothing on the board |
| Augmented-Lagrangian refit of `W`, `H2_I` with a fixed sign pattern (L-BFGS, tight tolerances) | within 1e-7 of the start |
| Our version-9 margin push and reduced-gradient descent (zero weights held at zero, signs fixed) | on Atakan Aldemir's `bac69f27`: -1.7e-8 (+0.0000002 on the board). prvsiyan's split-L1 refit moves the zero set and finds far more (section 7) |
| Cutoff 19 on the DQI | not shipped: a fresh draw with 0.354 expected excluded shots per 102,400 (section 3), worse in expectation on the private split |
| Moves between idle hosts (`max_hops=16`) on the DQI | nothing beyond the 241 retained products (section 4) |
| An extra host qubit on the DQI | 132 more products, far below the about 860 Toffolis one qubit costs (section 4) |
| Scratch-qubit reallocation on the DQI | `min_qubits` equals `qubits` (section 6) |
| Retention in the invsqrt | version 6: it needs an extra qubit for any saving |

The rest of the gap to the top of the board (about 29.9) is not in local rewrites of these circuits. It needs new constructions: a cheaper DQI decoder, a better chemistry factorisation, or more of the qubit-for-Toffoli trade that made this invsqrt.

## Credits and licences

**The invsqrt is `1ba197b0`**: James Barlow's (manwithacat, *The Real $150K Was the Friends We Made*) one-Toffoli-lighter version of the public 46-qubit line, as shipped with Asterios Panoras's dead-write fix in his *QCC 27.3122 | Frontier + Finer Chemistry Refit*. The fix itself is Asterios Panoras's (*QCC 27.30 | Public Frontier + Free CZ Fix in invsq*: a Toffoli whose target is only measured in the X basis becomes a classically controlled CZ) of Atakan Aldemir's `dcde7078` (*Quantum Circuits: Iso-Gap + DQI h5*): his cutting-plane refit of the 33 rotation angles of Ali Haydar Özdağ's `4560ac0c` (*QCC | submit - verify and package*), which replaces the 128-row coefficient table with a 64-row one and re-synthesises both table loaders on an open fork of Syed Asad Ali's Apache-2.0 generator. **The DQI decoder and the generator under the invsqrt are Syed Asad Ali's.** His *VQCC Own-Circuit Portfolio* v12 (dataset, Apache-2.0) and the companion *VQCC Own-Circuit Portfolio Notebook* contain the circuits and their complete source builders; version 5 of this notebook rebuilt all of his files from that source, byte for byte. In his words, the work was done with a team of AI agents and with Claude Science supporting the research. Copyright 2026 Syed Asad Ali, Apache-2.0; as he asks, any work built on this should credit the lineage starting from Syed Asad Ali.

**The chemistry circuits are maoxin123's**: the coded-word table and 4,033-Toffoli loaders from *QCC v27 | 4033-Toffoli coded-word chemistry* (their dataset `qcc-chemistry-4033-toffoli-loaders`, CC0), loaders and angle words byte for byte. **The start weights** are Atakan Aldemir's `e52b7cdd` for the 4033 table, refit for 12,000 iterations in our version 13 (`764dfcbd`) and continued here (*QCC: Gap-3 Residue Repair and C43 Native Compiler*). **Section 7 runs prvsiyan's `refit_grg.py` and `chemlib.py`** (dataset *Circuits That Survived the Salt*, Apache-2.0; notebook *Verifier-Guided Quantum Circuit Optimization*), with only the margins, the acceptance threshold and the run length changed by us. On the 4038 table, Heliosli's `2f0cb894` (*QCC circuits live*) was the start of our version 10 run. alex chilton's *Verified Three-Task Portfolio* keeps the public record of which file came from where. maoxin123's chemistry lineage, as they credit it: ozertuu's `quantum-sim-linalg-cache` table line and prvsiyan's refit; loader design after Syed Asad Ali, Lera (marculera)'s DF-THC pipeline and word codes, Ahmed Berat Özer's progressive rounding and Atakan Aldemir's π-gauge; James Barlow (manwithacat)'s `dfthc_refit`; busyaprime and Heliosli.

**The extractor-cutoff wrapper** is Ertuğrul Özer's (ozertuu), from his notebook *Quantum AI 27.083+ VQCC: DQI cutoff 17*, with his excluded-probability analysis; this version applies the same wrapper with cutoff 18.

**Ours:** the value-fingerprint scanner, idle-window analysis, retention planner and rewrite, the exhaustive self-test, the paired replay, the reallocation and peephole audits, the cutoff ladder, the 241 retained products, the version-9 multiplier analysis and margin push (`refit2.py`, Apache-2.0), and the margin and run-length settings of section 7.

**Earlier versions of this notebook:** version 13 (27.30692) and version 12 (27.30687) are this refit on the `e46d90c3` invsqrt after 12,000 and 3,000 iterations; versions 10 and 11 continued Heliosli's `2f0cb894` on the 4038 table (stopped when the 4033 table appeared); version 9 (27.29708) is this portfolio with Atakan Aldemir's weights `bac69f27` and our margin push; version 8 (27.11042) is version 7's portfolio with the cutoff-18 DQI and maoxin123's coded-word chemistry; version 7 (27.08903) is Syed Asad Ali's portfolio with Özer's cutoff 17 and our retention; version 6 (27.08407) is his portfolio with our retention; versions 3 and 4 (25.40767, 25.44957) used maoxin123's 392k DQI decoder (from `cd71c482` by team *i want to run gemma 9b*, Oleg Sher, Heliosli, Syed Asad Ali, SomeJoint and jday96314, released by ozertuu), the `9c708f7b` invsqrt (Asterios Panoras, WCao24, Thomas Gamet, manwithacat, gengsr, maoxin123 and the lineage they credit), and maoxin123's 4,089-Toffoli chemistry loaders with prvsiyan's refit and Lera (marculera)'s DF-THC pipeline.

**Reading the leaderboard draw:** busyaprime (Akmal Xodarev), manwithacat (James Barlow), prvsiyan. **Independent verification** of our files: alex chilton's *Verified Three-Task Portfolio*, maoxin123's *QCC v25*. **Tasks, kickmix and verifiers:** Google Quantum AI and Qualtran (Khattar et al., arXiv:2610.23832).

If we have missed or misattributed anyone, please say so in the comments and we will fix it.